# CANN 调试实验

## 8.1 实验概览

按顺序运行下面的代码，需要修改的参数已在代码中标出。


In [ ]:
%%bash
set -e
command -v bisheng mssanitizer msdebug


<a id="s-08-02_msdebug_mssanitizer"></a>


## 8.2 msSanitizer 与 msDebug

### 小节概述

先运行一个已知故障，阅读工具给出的诊断和源码位置，再比较其他故障。
故障程序可能返回非零；只有日志中出现对应诊断，才算观察到了题目设置的问题。
本章脚本负责构建、保存日志和检查诊断，源码仍是下面介绍的两种工程。


### 1. 故障代码

<table style="text-align:left; margin-left:0;">
<tr><th>MODE</th><th>故障设计</th><th>应观察到的诊断</th></tr>
<tr><td>memcheck</td><td>把 2048 个 float 的 DataCopy 改成 4096 个</td><td>illegal write</td></tr>
<tr><td>racecheck</td><td>删除 MTE2→Vector 的同步</td><td>Potential RAW hazard</td></tr>
<tr><td>initcheck</td><td>未初始化 zLocal 就搬到 GM</td><td>uninitialized read</td></tr>
<tr><td>synccheck</td><td>增加无 WaitFlag 的 SetFlag</td><td>Unpaired set_flag</td></tr>
</table>

### 先看搬运与计算

`GM（设备内存） --MTE2搬入--> UB（片上缓冲） --Vector计算--> UB --MTE3搬出--> GM`

Vector 读取 xLocal/yLocal 前要等待搬入完成。RAW 表示“写入完成之前发生读取”；SetFlag 发出完成事件，WaitFlag 等待对应事件。


完整源码位于 `src/add_sanitizer.asc`。以下摘录对应当前故障分支：

```cpp
// memcheck：xLocal 只有 blockLength 个元素，却搬入两倍长度。
AscendC::DataCopy(xLocal, xGm, blockLength * 2);

// racecheck：不执行下面这对同步，Vector 可能先读尚未搬完的数据。
AscendC::SetFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID0);
AscendC::WaitFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID0);

// initcheck：跳过 Add，直接读出尚未写入的 zLocal。
AscendC::DataCopy(zGm, zLocal, blockLength);

// synccheck：保留正常的 EVENT_ID0 同步，再加没有 WaitFlag 配对的 EVENT_ID1。
AscendC::SetFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID1);
```

racecheck 摘录展示的是被条件编译排除的两行，实际故障分支不执行它们。`LAB05_FAULT_MODE` 选择对应分支。


### 2. 独立 ASC 可执行程序

先编译一个越界故障程序。`LAB05_FAULT_MODE=memcheck` 选择源码中的越界分支，`LAB05_ENABLE_SANITIZER=ON` 让 CMake 添加 `--sanitizer` 插桩选项。


In [ ]:
%%bash
set -e
cmake -S src -B build/standalone-memcheck -DLAB05_FAULT_MODE=memcheck -DLAB05_ENABLE_SANITIZER=ON
cmake --build build/standalone-memcheck -j


再用 `mssanitizer` 运行程序并保存日志。`PIPESTATUS[0]` 取得工具的退出码；是否检测到越界还要看日志中的诊断。


In [ ]:
%%bash
BUILD=build/standalone-memcheck
mssanitizer --tool=memcheck "$BUILD/lab05_add_sanitizer" 2>&1 | tee "$BUILD/mssanitizer.log"
rc=${PIPESTATUS[0]}
echo "工具退出码：$rc"
set -e
grep -Ei 'illegal (read|write)|out[- ]of[- ]bounds' "$BUILD/mssanitizer.log"
echo 'SANITIZER_RESULT status=EXPECTED_DIAGNOSTIC mode=memcheck'


在输出中找到 `illegal read/write` 或越界诊断，对照源码解释原因。日志保存在 `build/standalone-memcheck/mssanitizer.log`。

再运行其余三种故障，分别观察数据竞争、未初始化读取和同步错误：


In [ ]:
%%bash
set -e
for mode in racecheck initcheck synccheck; do
  BUILD=build/standalone-$mode
  cmake -S src -B "$BUILD" -DLAB05_FAULT_MODE="$mode" -DLAB05_ENABLE_SANITIZER=ON
  cmake --build "$BUILD" -j
  # 故障程序可能非零退出，先保存实际退出码，再检查诊断。
  set +e
  mssanitizer --tool="$mode" "$BUILD/lab05_add_sanitizer" 2>&1 | tee "$BUILD/mssanitizer.log"
  rc=${PIPESTATUS[0]}
  set -e
  echo "$mode 退出码：$rc"
done
grep -Ei 'RAW hazard|data race' build/standalone-racecheck/mssanitizer.log
grep -Ei 'uninitialized' build/standalone-initcheck/mssanitizer.log
grep -Ei 'unpaired[ _-]*set_flag' build/standalone-synccheck/mssanitizer.log


最后运行没有故障、没有插桩的 baseline，检查程序本身能否正常计算。


In [ ]:
%%bash
set -e
cmake -S src -B build/standalone-baseline -DLAB05_FAULT_MODE=baseline -DLAB05_ENABLE_SANITIZER=OFF
cmake --build build/standalone-baseline -j
./build/standalone-baseline/lab05_add_sanitizer
echo 'BASELINE_STATUS=PASS'


### 3. 标准算子工程

标准工程的 Kernel 目标由 CANN 构建系统创建，因此采用当前版本提供的辅助函数：

```cmake
if(COMMAND add_ops_compile_options)
  add_ops_compile_options(ALL OPTIONS -sanitizer)
elseif(COMMAND npu_op_kernel_options)
  npu_op_kernel_options(ascendc_kernels ALL OPTIONS -sanitizer)
endif()
```

两者都把 `-sanitizer` 送入 Kernel 编译；区别只是 CANN 版本提供的函数名不同。应把插桩选项加到 Kernel 的编译目标。

独立 ASC 是一个直接启动 Kernel 的程序；标准算子工程还包含算子包和配套 Host 调用，学习的是同一个故障如何经过不同构建、调用流程被发现。

标准工程应修改 `src/operator_project_sanitizer/op_kernel/add_custom_msanitizer.cpp.in`。日志中的 `build-模式/lab05_generated_kernel/` 是当前构建生成的源码，用来对应行号，重新构建时会被覆盖。


标准工程把 `LAB05_OPERATOR_FAULT_MODE` 转成 Kernel 编译定义，并在开启实验插桩时调用上述辅助函数。完整配置位于 `src/operator_project_sanitizer/op_kernel/CMakeLists.txt`；无需在 Notebook 中展开整份文件。

下面依次运行标准算子工程中的四种故障。[run_sanitizer.sh](src/run_sanitizer.sh) 负责构建、临时安装、运行和清理，并检查对应诊断。

日志保存在 `src/operator_project_sanitizer/build-模式/mssanitizer.log`。


In [ ]:
%%bash
set -e
for mode in memcheck racecheck initcheck synccheck; do
  bash src/run_sanitizer.sh operator "$mode"
done


### 4. msDebug

使用正常程序观察断点和调用栈。先检查调试开关和设备权限；满足条件后才编译和启动调试。条件不足时记录 `BLOCKED`，后面的两个单元格会跳过调试操作。重测时从这个检查单元格开始。


阅读主线是“设断点→运行→看栈帧与源码→继续”。下面的权限、状态和日志检查格直接运行即可，无需修改。

| 调试命令 | 观察内容 |
|---|---|
| breakpoint set / run | 在 Add 所在源码行停下 |
| thread backtrace / frame info | 当前设备调用栈和栈帧 |
| source list | 当前执行位置对应的源码 |
| breakpoint disable / continue | 关闭断点，继续到程序结束 |


In [ ]:
%%bash
set -e
mkdir -p build/msdebug
echo BLOCKED > build/msdebug/status.txt
: > build/msdebug/msdebug.log
if ! command -v msdebug; then
  echo 'MSDEBUG_STATUS=BLOCKED reason=msdebug_missing'
  exit 0
fi
if [[ ! -r /proc/debug_switch || ! -r /dev/drv_debug || ! -w /dev/drv_debug ]]; then
  echo 'MSDEBUG_STATUS=BLOCKED reason=debug_permission_missing'
  exit 0
fi
cat /proc/debug_switch
# 兼容 1 和 debug_switch_status = 1 两种输出。
if ! grep -Eq '(^|=[[:space:]]*)1[[:space:]]*$' /proc/debug_switch; then
  echo 'MSDEBUG_STATUS=BLOCKED reason=debug_switch_disabled'
  exit 0
fi
echo FAIL > build/msdebug/status.txt
cmake -S src -B build/msdebug -DLAB05_FAULT_MODE=baseline -DLAB05_ENABLE_SANITIZER=OFF
cmake --build build/msdebug -j
echo READY > build/msdebug/status.txt


检查通过后，下面将调试命令写入文件。断点设在 `add_sanitizer.asc` 第 68 行的 `AscendC::Add`；随后运行程序，查看调用栈、当前栈帧和源码，再继续到结束。


In [ ]:
%%bash
set -e
state=$(cat build/msdebug/status.txt)
if [[ "$state" != READY ]]; then
  echo "MSDEBUG_STATUS=$state action=SKIPPED"
  exit 0
fi
cat > build/msdebug/msdebug.cmd <<'EOF'
breakpoint set --file add_sanitizer.asc --line 68
run
thread backtrace
frame info
source list
breakpoint disable 1
continue
quit
EOF


执行调试命令，日志保存到 `build/msdebug/msdebug.log`。检查实际停止原因、当前设备栈帧和源码，再确认程序正常结束。`Breakpoint 1: where ...` 只表示断点已设置，不能据此判断命中。


In [ ]:
%%bash
set -eo pipefail
state=$(cat build/msdebug/status.txt)
if [[ "$state" != READY ]]; then
  echo "MSDEBUG_STATUS=$state action=SKIPPED"
  exit 0
fi
echo FAIL > build/msdebug/status.txt
set +e
printf 'y\n' | timeout 300 msdebug --batch --source build/msdebug/msdebug.cmd -- \
  build/msdebug/lab05_add_sanitizer 2>&1 | tee build/msdebug/msdebug.log
rc=${PIPESTATUS[1]}
set -e
LOG=build/msdebug/msdebug.log
if grep -Fqi 'msdebug failed to initialize' "$LOG"; then
  echo BLOCKED > build/msdebug/status.txt
  echo 'MSDEBUG_STATUS=BLOCKED reason=debugger_initialization_failed'
  exit 0
fi
if [[ "$rc" -ne 0 ]] || grep -Eq 'stop reason = signal|exited with status = [1-9]' "$LOG"; then
  echo "MSDEBUG_STATUS=FAIL reason=abnormal_exit exit_code=$rc"
  exit 1
fi
if ! grep -Eq 'stop reason = breakpoint' "$LOG" ||
   ! grep -Eq 'frame #0:.*add_custom.*add_sanitizer\.asc:68(:|[[:space:]]|$)' "$LOG" ||
   ! grep -Fq 'AscendC::Add(' "$LOG" ||
   ! grep -Eq 'Process [0-9]+ exited with status = 0' "$LOG" ||
   ! grep -Fiq 'test pass' "$LOG"; then
  echo 'MSDEBUG_STATUS=FAIL reason=debug_evidence_incomplete'
  exit 1
fi
echo PASS > build/msdebug/status.txt
echo 'MSDEBUG_STATUS=PASS'


断点命中后应显示设备栈帧和对应源码；继续运行后，程序应正常结束并输出 `test pass`。平台未开启调试或权限不足时显示 `BLOCKED`，联系平台管理员后再测。


### 课后实践

任选 racecheck、initcheck 或 synccheck，在两种工程下比较诊断，指出问题代码、源码行号和原因。


完成本节练习后查看：[参考答案](answer/08.02_answer.md)